# Hybrid Index Metadata Linking + Deletion

A hybrid retrieval system keeps two separate indexes over the same documents:

- a **BM25 lexical index** (keyword match)
- an **embedding index** (semantic match, TF-IDF vectors stand in for a real embedding model here)

Both indexes are built from the same document list, so a `doc_id` at position `i` in one index refers to the same document as position `i` in the other. A **metadata store** (`doc_id -> subject_id`) is the link between them and the outside world.

When a deletion request comes in for a `subject_id` (e.g. a GDPR erasure request), the orchestrator:
1. looks up every `doc_id` owned by that `subject_id` in the metadata store,
2. removes those documents, then
3. rebuilds both indexes so neither the lexical nor the embedding index can surface the deleted content again.

Run the cells in order.

In [ ]:
%pip install rank_bm25 scikit-learn numpy

In [ ]:
import re
from dataclasses import dataclass

import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


@dataclass
class Document:
    doc_id: str
    subject_id: str
    text: str

In [ ]:
class HybridIndex:
    """Keeps a BM25 lexical index and a TF-IDF embedding index in sync via a
    shared metadata store (doc_id -> subject_id), so a delete request can
    fan out across both indexes plus the metadata itself."""

    def __init__(self):
        self.metadata: dict[str, Document] = {}
        self.doc_ids: list[str] = []
        self.bm25 = None
        self.embeddings = None
        self._vectorizer = TfidfVectorizer()

    def add(self, doc: Document) -> None:
        self.metadata[doc.doc_id] = doc
        self._rebuild()

    def delete_by_subject_id(self, subject_id: str) -> list[str]:
        """GDPR-style erasure: find every doc owned by subject_id, remove it
        from the metadata store, then rebuild both indexes so the deleted
        content can no longer be retrieved from either one."""
        removed = [doc_id for doc_id, doc in self.metadata.items() if doc.subject_id == subject_id]
        for doc_id in removed:
            del self.metadata[doc_id]
        self._rebuild()
        return removed

    def _rebuild(self) -> None:
        self.doc_ids = list(self.metadata.keys())
        texts = [self.metadata[doc_id].text for doc_id in self.doc_ids]

        # Lexical index: rebuilt from the surviving documents only
        self.bm25 = BM25Okapi([tokenize(t) for t in texts]) if texts else None

        # Embedding index: rebuilt from the surviving documents only
        if texts:
            self._vectorizer = TfidfVectorizer()
            self.embeddings = self._vectorizer.fit_transform(texts)
        else:
            self.embeddings = None

    def search(self, query: str, k: int = 3, bm25_weight: float = 0.5) -> list[dict]:
        if not self.doc_ids:
            return []

        bm25_scores = np.array(self.bm25.get_scores(tokenize(query)))
        bm25_scores = bm25_scores / (bm25_scores.max() or 1)

        query_vec = self._vectorizer.transform([query])
        embed_scores = cosine_similarity(query_vec, self.embeddings).flatten()

        combined = bm25_weight * bm25_scores + (1 - bm25_weight) * embed_scores
        ranked_idx = combined.argsort()[::-1][:k]

        return [
            {
                "doc_id": self.doc_ids[i],
                "subject_id": self.metadata[self.doc_ids[i]].subject_id,
                "text": self.metadata[self.doc_ids[i]].text,
                "score": round(float(combined[i]), 4),
            }
            for i in ranked_idx
        ]

In [ ]:
# Two documents belong to subject "user_42", one to "user_7", one to "user_9"
index = HybridIndex()
for doc in [
    Document("doc-1", "user_42", "Invoice for consulting services rendered in March"),
    Document("doc-2", "user_42", "Support ticket about a delayed invoice payment"),
    Document("doc-7", "user_7", "Contract renewal terms for cloud storage subscription"),
    Document("doc-9", "user_9", "Refund request for a cancelled invoice"),
]:
    index.add(doc)

print("Metadata store:")
for doc_id, doc in index.metadata.items():
    print(f"  {doc_id} -> subject={doc.subject_id}")

In [ ]:
print("Search results BEFORE deletion:")
for hit in index.search("invoice payment"):
    print(f"  [{hit['score']:.4f}] {hit['doc_id']} (subject={hit['subject_id']}): {hit['text']}")

In [ ]:
# Erasure request: delete every document belonging to user_42
removed_doc_ids = index.delete_by_subject_id("user_42")
print(f"Deleted from BM25 index, embedding index, and metadata store: {removed_doc_ids}")

In [ ]:
print("Search results AFTER deletion (user_42's docs must not appear):")
for hit in index.search("invoice payment"):
    print(f"  [{hit['score']:.4f}] {hit['doc_id']} (subject={hit['subject_id']}): {hit['text']}")

assert "user_42" not in {doc.subject_id for doc in index.metadata.values()}
print("\nVerified: user_42 is gone from the metadata store.")